# Stage 1 — XGBoost (Barrier Prediction)

Train and evaluate XGBoost classifiers for three independent Stage 1 barrier targets:

- `target_household`
- `target_logistic`
- `target_facility`

Uses processed NFHS-5 individual-level data and the shared `split_and_scale()` pipeline.

## 1. Load Libraries

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.xgboost_model import (
    TARGET_KEYS,
    XGB_HYPERPARAMETERS,
    configure_logging,
    load_processed_data,
    split_target_data,
    train_xgboost,
)
from src.results.xgboost_results import (
    evaluate_xgboost_model,
    plot_confusion_matrix_heatmap,
    plot_feature_importance,
    plot_roc_curve,
    save_all_xgboost_outputs,
)

configure_logging()
print(f"Project root: {PROJECT_ROOT}")
print(f"Hyperparameters: {XGB_HYPERPARAMETERS}")

Project root: C:\major project\BarrierLens_MP_G25_P48
Hyperparameters: {'n_estimators': 400, 'max_depth': 6, 'learning_rate': 0.08, 'subsample': 0.8, 'colsample_bytree': 0.8, 'tree_method': 'hist', 'eval_metric': 'auc', 'random_state': 42, 'n_jobs': -1}


## 2. Load Processed Dataset

In [2]:
processed_dir = PROJECT_ROOT / "data" / "processed"
X, targets = load_processed_data(processed_dir)

print(f"X shape: {X.shape}")
for key in TARGET_KEYS:
    y = targets[key]
    print(f"\n{key}: n={len(y):,}, positive rate={y.mean():.4f}")
    print(y.value_counts().sort_index())

2026-07-20 16:55:18,277 | INFO | src.models.xgboost_model | Loaded processed data — X shape: (724115, 37)


X shape: (724115, 37)

household: n=724,115, positive rate=0.2716
target_household
0    527477
1    196638
Name: count, dtype: int64

logistic: n=724,115, positive rate=0.3161
target_logistic
0    495248
1    228867
Name: count, dtype: int64

facility: n=724,115, positive rate=0.4601
target_facility
0    390968
1    333147
Name: count, dtype: int64


## 3. Split Data

In [3]:
splits = {key: split_target_data(X, targets[key], key) for key in TARGET_KEYS}

for key, split_data in splits.items():
    print(
        f"{key}: train={len(split_data['y_train']):,}, "
        f"test={len(split_data['y_test']):,}, "
        f"scale_pos_weight={split_data['scale_pos_weight']:.4f}"
    )

target_household — Train: 579,292 rows | Test: 144,823 rows | train positive rate: 0.2716


2026-07-20 16:55:23,796 | INFO | src.models.xgboost_model | Class balance — negatives: 421982, positives: 157310, scale_pos_weight: 2.6825


target_logistic — Train: 579,292 rows | Test: 144,823 rows | train positive rate: 0.3161


2026-07-20 16:55:28,082 | INFO | src.models.xgboost_model | Class balance — negatives: 396198, positives: 183094, scale_pos_weight: 2.1639


target_facility — Train: 579,292 rows | Test: 144,823 rows | train positive rate: 0.4601


2026-07-20 16:55:31,264 | INFO | src.models.xgboost_model | Class balance — negatives: 312774, positives: 266518, scale_pos_weight: 1.1736


household: train=579,292, test=144,823, scale_pos_weight=2.6825
logistic: train=579,292, test=144,823, scale_pos_weight=2.1639
facility: train=579,292, test=144,823, scale_pos_weight=1.1736


## 4. Train XGBoost

In [4]:
models_dir = PROJECT_ROOT / "models"
models = {}

for key in TARGET_KEYS:
    split_data = splits[key]
    models[key] = train_xgboost(
        split_data["X_train"],
        split_data["y_train"],
        key,
        save_path=models_dir / f"xgb_{key}.pkl",
    )

print("Saved models:")
for path in sorted(models_dir.glob("xgb_*.pkl")):
    print(f"  - {path.name}")

2026-07-20 16:55:31,775 | INFO | src.models.xgboost_model | Class balance — negatives: 421982, positives: 157310, scale_pos_weight: 2.6825
2026-07-20 16:55:31,807 | INFO | src.models.xgboost_model | Training XGBoost for target_household ...
2026-07-20 16:56:03,527 | INFO | src.models.xgboost_model | Finished training XGBoost for target_household.
2026-07-20 16:56:03,788 | INFO | src.models.xgboost_model | Saved model to C:\major project\BarrierLens_MP_G25_P48\models\xgb_household.pkl
2026-07-20 16:56:04,091 | INFO | src.models.xgboost_model | Class balance — negatives: 396198, positives: 183094, scale_pos_weight: 2.1639
2026-07-20 16:56:04,117 | INFO | src.models.xgboost_model | Training XGBoost for target_logistic ...
2026-07-20 16:56:41,495 | INFO | src.models.xgboost_model | Finished training XGBoost for target_logistic.
2026-07-20 16:56:41,617 | INFO | src.models.xgboost_model | Saved model to C:\major project\BarrierLens_MP_G25_P48\models\xgb_logistic.pkl
2026-07-20 16:56:41,625 |

Saved models:
  - xgb_facility.pkl
  - xgb_household.pkl
  - xgb_logistic.pkl


## 5. Evaluate

In [5]:
feature_names = X.columns.tolist()
evaluation = {}

for key in TARGET_KEYS:
    result = evaluate_xgboost_model(
        models[key],
        splits[key]["X_test"],
        splits[key]["y_test"],
        key,
    )
    evaluation[key] = result
    print(f"\n=== {key} ===")
    for metric, value in result["metrics"].items():
        print(f"  {metric}: {value}")
    print(result["classification_report"])

2026-07-20 16:57:17,358 | INFO | src.results.xgboost_results | XGBoost | household — Accuracy: 0.6015, ROC-AUC: 0.6619, F1: 0.4701



=== household ===
  Model: XGBoost
  Target: household
  Accuracy: 0.6015
  Precision: 0.3679
  Recall: 0.6507
  F1-Score: 0.4701
  ROC-AUC: 0.6619
              precision    recall  f1-score   support

           0       0.82      0.58      0.68    105495
           1       0.37      0.65      0.47     39328

    accuracy                           0.60    144823
   macro avg       0.59      0.62      0.58    144823
weighted avg       0.70      0.60      0.62    144823



2026-07-20 16:57:18,671 | INFO | src.results.xgboost_results | XGBoost | logistic — Accuracy: 0.6055, ROC-AUC: 0.6696, F1: 0.5184



=== logistic ===
  Model: XGBoost
  Target: logistic
  Accuracy: 0.6055
  Precision: 0.422
  Recall: 0.6718
  F1-Score: 0.5184
  ROC-AUC: 0.6696
              precision    recall  f1-score   support

           0       0.79      0.57      0.67     99050
           1       0.42      0.67      0.52     45773

    accuracy                           0.61    144823
   macro avg       0.61      0.62      0.59    144823
weighted avg       0.67      0.61      0.62    144823



2026-07-20 16:57:20,051 | INFO | src.results.xgboost_results | XGBoost | facility — Accuracy: 0.5803, ROC-AUC: 0.6185, F1: 0.5736



=== facility ===
  Model: XGBoost
  Target: facility
  Accuracy: 0.5803
  Precision: 0.5385
  Recall: 0.6136
  F1-Score: 0.5736
  ROC-AUC: 0.6185
              precision    recall  f1-score   support

           0       0.63      0.55      0.59     78194
           1       0.54      0.61      0.57     66629

    accuracy                           0.58    144823
   macro avg       0.58      0.58      0.58    144823
weighted avg       0.59      0.58      0.58    144823



## 6. Plot Graphs

In [6]:
plots_dir = PROJECT_ROOT / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

for key in TARGET_KEYS:
    result = evaluation[key]
    plot_roc_curve(
        splits[key]["y_test"],
        result["y_prob"],
        key,
        plots_dir,
    )
    plot_confusion_matrix_heatmap(
        result["confusion_matrix"],
        key,
        plots_dir,
    )
    plot_feature_importance(models[key], feature_names, key, plots_dir)

print("Saved plots:")
for path in sorted(plots_dir.glob("xgb_*.png")):
    print(f"  - {path.name}")

2026-07-20 16:57:23,234 | INFO | src.results.xgboost_results | Saved ROC plot to C:\major project\BarrierLens_MP_G25_P48\plots\xgb_household_roc.png
2026-07-20 16:57:24,038 | INFO | src.results.xgboost_results | Saved confusion matrix plot to C:\major project\BarrierLens_MP_G25_P48\plots\xgb_household_confusion.png
2026-07-20 16:57:24,868 | INFO | src.results.xgboost_results | Saved feature importance plot to C:\major project\BarrierLens_MP_G25_P48\plots\xgb_household_importance.png
2026-07-20 16:57:25,448 | INFO | src.results.xgboost_results | Saved ROC plot to C:\major project\BarrierLens_MP_G25_P48\plots\xgb_logistic_roc.png
2026-07-20 16:57:26,067 | INFO | src.results.xgboost_results | Saved confusion matrix plot to C:\major project\BarrierLens_MP_G25_P48\plots\xgb_logistic_confusion.png
2026-07-20 16:57:26,786 | INFO | src.results.xgboost_results | Saved feature importance plot to C:\major project\BarrierLens_MP_G25_P48\plots\xgb_logistic_importance.png
2026-07-20 16:57:27,401 | I

Saved plots:
  - xgb_facility_confusion.png
  - xgb_facility_importance.png
  - xgb_facility_roc.png
  - xgb_household_confusion.png
  - xgb_household_importance.png
  - xgb_household_roc.png
  - xgb_logistic_confusion.png
  - xgb_logistic_importance.png
  - xgb_logistic_roc.png


## 7. Save Outputs

In [7]:
dataset_stats = {
    "n_samples": len(X),
    "n_features": X.shape[1],
    "targets": {
        key: {
            "negative": int((targets[key] == 0).sum()),
            "positive": int((targets[key] == 1).sum()),
            "positive_rate": round(float(targets[key].mean()), 4),
            "scale_pos_weight": splits[key]["scale_pos_weight"],
            "train_size": len(splits[key]["y_train"]),
            "test_size": len(splits[key]["y_test"]),
        }
        for key in TARGET_KEYS
    },
}
training_bundle = {
    "feature_names": feature_names,
    "splits": splits,
    "models": models,
    "hyperparameters": dict(XGB_HYPERPARAMETERS),
    "dataset_stats": dataset_stats,
    "models_dir": models_dir,
    "processed_dir": processed_dir,
}
metrics_df = save_all_xgboost_outputs(training_bundle)

display(metrics_df)
print(f"\nReport: {PROJECT_ROOT / 'results' / 'xgboost_report.md'}")

2026-07-20 16:57:29,950 | INFO | src.results.xgboost_results | XGBoost | household — Accuracy: 0.6015, ROC-AUC: 0.6619, F1: 0.4701
2026-07-20 16:57:31,022 | INFO | src.results.xgboost_results | Saved predictions to C:\major project\BarrierLens_MP_G25_P48\results\xgb_predictions_household.csv
2026-07-20 16:57:31,490 | INFO | src.results.xgboost_results | Saved ROC plot to C:\major project\BarrierLens_MP_G25_P48\plots\xgb_household_roc.png
2026-07-20 16:57:31,806 | INFO | src.results.xgboost_results | Saved confusion matrix plot to C:\major project\BarrierLens_MP_G25_P48\plots\xgb_household_confusion.png
2026-07-20 16:57:32,339 | INFO | src.results.xgboost_results | Saved feature importance plot to C:\major project\BarrierLens_MP_G25_P48\plots\xgb_household_importance.png
2026-07-20 16:57:34,759 | INFO | src.results.xgboost_results | XGBoost | logistic — Accuracy: 0.6055, ROC-AUC: 0.6696, F1: 0.5184
2026-07-20 16:57:35,129 | INFO | src.results.xgboost_results | Saved predictions to C:\ma

,Model,Target,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,XGBoost,household,0.6015,0.3679,0.6507,0.4701,0.6619
1,XGBoost,logistic,0.6055,0.4220,0.6718,0.5184,0.6696
2,XGBoost,facility,0.5803,0.5385,0.6136,0.5736,0.6185



Report: C:\major project\BarrierLens_MP_G25_P48\results\xgboost_report.md


## 8. Final Observations

In [8]:
best = metrics_df.sort_values("ROC-AUC", ascending=False).iloc[0]
print(
    f"Best target: {best['Target']} (ROC-AUC = {best['ROC-AUC']:.4f}, "
    f"F1 = {best['F1-Score']:.4f})"
)
print("\nKey takeaways:")
print("- XGBoost uses scale_pos_weight per target to handle class imbalance.")
print("- Household barriers are typically easiest to predict; facility barriers are hardest.")
print("- All artefacts are saved under models/, results/, and plots/ for notebook 06.")

Best target: logistic (ROC-AUC = 0.6696, F1 = 0.5184)

Key takeaways:
- XGBoost uses scale_pos_weight per target to handle class imbalance.
- Household barriers are typically easiest to predict; facility barriers are hardest.
- All artefacts are saved under models/, results/, and plots/ for notebook 06.
